# Pandas — Phase 6: Merging, Joining, and Date Manipulation
### Credit Card Risk Analysis Track

**Topics in this phase:**
25. Concatenation — `pd.concat`
26. Merging DataFrames — `pd.merge`
27. Datetime Conversion — `pd.to_datetime`
28. Date Feature Extraction — pulling date parts, elapsed time

**New datasets for this phase** (place all in the same folder as this notebook):
- `loan_batch_jan.csv`, `loan_batch_feb.csv`, `loan_batch_mar.csv` — the same loan applications split into 3 pretend "monthly export" files, for the concatenation topic.
- `credit_bureau.csv` — a separate external file: bureau score, inquiry count, open credit lines, and last delinquency date, keyed by `application_id`. It covers 470 of the 500 loan applicants (30 have no bureau hit) plus 20 bureau records for people who never applied for this loan product — deliberately imperfect overlap, so the different join types actually behave differently.

**How to use this notebook:**
- Each question has a `YOUR CODE HERE` cell — attempt it first.
- The `Solution` cell right after shows one correct approach — compare, don't just copy.
- All solutions were run against the actual datasets before this notebook was assembled.

## Setup

In [ ]:
import pandas as pd
import numpy as np
print(pd.__version__)

## Topic 25: Concatenation

Real loan data often arrives as separate monthly (or daily) exports. `pd.concat` stacks them back into one table. Run this setup cell first:

In [ ]:
batch_jan = pd.read_csv("loan_batch_jan.csv", parse_dates=["application_date"])
batch_feb = pd.read_csv("loan_batch_feb.csv", parse_dates=["application_date"])
batch_mar = pd.read_csv("loan_batch_mar.csv", parse_dates=["application_date"])
print(batch_jan.shape, batch_feb.shape, batch_mar.shape)

**Q1.** Stack all three monthly batches into one DataFrame `all_batches` using `pd.concat([...])`. Print the shape — it should have all the rows of the three batches combined.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
all_batches = pd.concat([batch_jan, batch_feb, batch_mar])
print(all_batches.shape)

**Q2.** Look at `all_batches.index` from Q1 — since each batch kept its own original `0, 1, 2, ...` index, the combined index has **repeats**. Redo the concat with `ignore_index=True`, into `all_batches_reset`, so the index is one continuous `0` to `n-1` sequence. Print the first and last 5 index values.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
all_batches_reset = pd.concat([batch_jan, batch_feb, batch_mar], ignore_index=True)
print(all_batches_reset.index[:5].tolist(), all_batches_reset.index[-5:].tolist())

**Q3.** Sometimes you want to know *which batch* each row came from after combining. Redo the concat using `keys=["Jan", "Feb", "Mar"]` and `names=["batch_month"]`, into `all_batches_tagged` — this adds an extra index level recording the source batch. Print the unique values of that new index level with `.index.get_level_values("batch_month").unique()`.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
all_batches_tagged = pd.concat(
    [batch_jan, batch_feb, batch_mar], keys=["Jan", "Feb", "Mar"], names=["batch_month"]
)
print(all_batches_tagged.index.get_level_values("batch_month").unique().tolist())

**Q4.** Sanity-check your concat: confirm that `len(batch_jan) + len(batch_feb) + len(batch_mar)` equals `len(all_batches)`, into a boolean `combined_shape_check`. This kind of row-count check is worth doing every time you concatenate real data exports, to catch a silently dropped or duplicated file.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
combined_shape_check = len(batch_jan) + len(batch_feb) + len(batch_mar) == len(all_batches)
print(combined_shape_check)

**Q5.** `pd.concat` isn't just for stacking rows — with `axis=1` it glues DataFrames **side by side** by their index instead. Given the two small DataFrames below (both indexed `0, 1`), combine them side by side into `side_by_side` using `pd.concat([...], axis=1)`. Notice the result has two `application_id` columns — `axis=1` concat doesn't check for or merge on a shared key the way `pd.merge` does (that's next in Topic 26).

In [ ]:
loan_columns_only = pd.DataFrame({"application_id": [1, 2], "loan_amount": [10000, 5000]})
bureau_columns_only = pd.DataFrame({"application_id": [1, 2], "bureau_score": [700, 650]})

# YOUR CODE HERE


**Solution**

In [ ]:
side_by_side = pd.concat([loan_columns_only, bureau_columns_only], axis=1)
print(side_by_side)

## Topic 26: Merging DataFrames

`pd.merge` combines two tables based on matching **key values** (like a SQL join), not just position — exactly what you need to attach an external credit bureau file to your applications. Run this setup cell first:

In [ ]:
df = pd.read_csv("loan_applications.csv", parse_dates=["application_date"])
credit_bureau = pd.read_csv("credit_bureau.csv")
print(df.shape, credit_bureau.shape)

**Q6.** Merge `df` and `credit_bureau` on `application_id` using `how="inner"` (only keep applications that have a bureau record **and** bureau records that match an application), into `merged_inner`. Print the shape — how many rows matched on both sides?

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
merged_inner = pd.merge(df, credit_bureau, on="application_id", how="inner")
print(merged_inner.shape)

**Q7.** Merge with `how="left"` instead, into `merged_left` — keep **every** loan application, filling bureau columns with `NaN` where there's no match. Print the shape and how many rows ended up with a missing `bureau_score` (the 30 applicants with no bureau hit).

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
merged_left = pd.merge(df, credit_bureau, on="application_id", how="left")
print(merged_left.shape, merged_left["bureau_score"].isna().sum())

**Q8.** Merge with `how="right"`, into `merged_right` — keep **every** bureau record, filling loan columns with `NaN` where there's no matching application (the 20 bureau records for non-applicants). Print the shape and how many rows have a missing `loan_amount`.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
merged_right = pd.merge(df, credit_bureau, on="application_id", how="right")
print(merged_right.shape, merged_right["loan_amount"].isna().sum())

**Q9.** Merge with `how="outer"`, into `merged_outer` — keep **everything** from both sides, matched where possible. Print the shape (it should be roughly `len(merged_inner) + unmatched-from-each-side`).

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
merged_outer = pd.merge(df, credit_bureau, on="application_id", how="outer")
print(merged_outer.shape)

**Q10.** Redo the outer merge with `indicator=True`, into `merged_indicator` — this adds a `_merge` column telling you exactly where each row came from. Get the value counts of `_merge` into `match_counts`: how many rows were `"both"`, `"left_only"`, and `"right_only"`? These should match your Q6-Q8 findings exactly.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
merged_indicator = pd.merge(df, credit_bureau, on="application_id", how="outer", indicator=True)
match_counts = merged_indicator["_merge"].value_counts()
print(match_counts)

**Q11.** You don't have to merge entire DataFrames — select just the columns you need from each side first. Merge just `application_id`+`loan_amount` from `df` with just `application_id`+`bureau_score` from `credit_bureau`, `how="inner"`, into `loan_amount_and_bureau`. This keeps the result lean when you don't need every column.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
loan_amount_and_bureau = pd.merge(
    df[["application_id", "loan_amount"]],
    credit_bureau[["application_id", "bureau_score"]],
    on="application_id",
    how="inner",
)
print(loan_amount_and_bureau.shape)

## Topic 27: Datetime Conversion

`last_delinquency_date` in `credit_bureau.csv` came in as plain text — `pd.to_datetime` is how you turn date-like strings into something you can actually do date math on.

**Q12.** Given `raw_dates` below (clean ISO-format strings), convert it to real datetimes into `parsed_dates` using `pd.to_datetime`. Print the resulting dtype.

In [ ]:
raw_dates = pd.Series(["2024-03-15", "2024-07-01", "2025-01-20"])

# YOUR CODE HERE


**Solution**

In [ ]:
parsed_dates = pd.to_datetime(raw_dates)
print(parsed_dates.dtype)

**Q13.** Real data is rarely this clean. Given `messy_dates` below — a mix of formats, plus one value that isn't a date at all — convert it with `pd.to_datetime`, passing `errors="coerce"` (unparseable values become `NaT` instead of crashing) and `format="mixed"` (lets pandas infer a different format per value), into `parsed_messy`. Print the result.

In [ ]:
messy_dates = pd.Series(["03/15/2024", "07-01-2024", "not a date"])

# YOUR CODE HERE


**Solution**

In [ ]:
parsed_messy = pd.to_datetime(messy_dates, errors="coerce", format="mixed")
print(parsed_messy.tolist())

**Q14.** Convert the `last_delinquency_date` column in `merged_left` (from Topic 26) to actual datetimes **in place**, using `pd.to_datetime`. Print the resulting dtype. (Missing values — applicants with no delinquency on file — should become `NaT`, pandas' "missing datetime" value, automatically.)

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
merged_left["last_delinquency_date"] = pd.to_datetime(merged_left["last_delinquency_date"])
print(merged_left["last_delinquency_date"].dtype)

**Q15.** Given `specific_format_dates` below in day-first format (`DD-MM-YYYY`), convert it correctly by passing an explicit `format="%d-%m-%Y"` — without it, pandas might guess month-first and silently get the wrong date. Store as `parsed_specific` and print the result.

In [ ]:
specific_format_dates = pd.Series(["15-03-2024", "01-07-2024"])

# YOUR CODE HERE


**Solution**

In [ ]:
parsed_specific = pd.to_datetime(specific_format_dates, format="%d-%m-%Y")
print(parsed_specific.tolist())

## Topic 28: Date Feature Extraction

Once a column is a real datetime, the `.dt` accessor unlocks components and arithmetic — this is where raw timestamps become model-ready features.

**Q16.** On `merged_left`, extract `application_year` and `application_month` from `application_date` using `.dt.year` and `.dt.month`.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
merged_left["application_year"] = merged_left["application_date"].dt.year
merged_left["application_month"] = merged_left["application_date"].dt.month
print(merged_left[["application_date", "application_year", "application_month"]].head(3))

**Q17.** Extract `application_day_of_week`, the name of the weekday (e.g. `"Monday"`), using `.dt.day_name()`.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
merged_left["application_day_of_week"] = merged_left["application_date"].dt.day_name()
print(merged_left["application_day_of_week"].head(5).tolist())

**Q18.** Compute `days_since_application`: how many days ago each application was submitted, relative to `today = pd.Timestamp("2026-07-14")`. Subtracting two datetimes gives you a `Timedelta` — chain `.dt.days` to get a plain integer count of days.

In [ ]:
today = pd.Timestamp("2026-07-14")

# YOUR CODE HERE


**Solution**

In [ ]:
today = pd.Timestamp("2026-07-14")
merged_left["days_since_application"] = (today - merged_left["application_date"]).dt.days
print(merged_left["days_since_application"].head(3).tolist())

**Q19.** This is the exact pattern from the phase brief: compute `days_since_last_delinquency` the same way, using `last_delinquency_date` (from Topic 27) instead. Print the first 10 values — you'll see `NaN` for applicants with no delinquency on record, since there's no date to subtract from.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
merged_left["days_since_last_delinquency"] = (
    today - merged_left["last_delinquency_date"]
).dt.days
print(merged_left["days_since_last_delinquency"].head(10).tolist())

**Q20.** Those `NaN`s in Q19 are meaningful, not missing data to clean up — they mean "never delinquent," which is good news, not a data quality problem. Create a boolean `never_delinquent` flag: `True` where `last_delinquency_date` is null, using `.isna()`. Print how many applicants that covers.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
merged_left["never_delinquent"] = merged_left["last_delinquency_date"].isna()
print(merged_left["never_delinquent"].sum())

**Q21.** Create `recently_delinquent`: `True` where `days_since_last_delinquency` is under 365 **and** the applicant has actually had a delinquency (`~never_delinquent`) — without that second condition, `NaN < 365` would silently evaluate to `False` and you'd get the right answer by accident here, but it's worth being explicit about the intent.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
merged_left["recently_delinquent"] = (
    (merged_left["days_since_last_delinquency"] < 365) & (~merged_left["never_delinquent"])
)
print(merged_left["recently_delinquent"].sum())

## ✅ Checkpoint

**What you covered:**
- Concatenation: stacking with `pd.concat`, `ignore_index=True` to fix repeated indices, `keys=` to tag rows by source, sanity-checking row counts, and side-by-side concat with `axis=1`
- Merging: `pd.merge` with `how="inner"`/`"left"`/`"right"`/`"outer"`, `indicator=True` to see exactly which rows matched, and merging on a subset of columns
- Datetime conversion: `pd.to_datetime` on clean strings, `errors="coerce"` + `format="mixed"` for messy real-world dates, and explicit `format=` strings to avoid day/month ambiguity
- Date feature extraction: `.dt.year`/`.dt.month`/`.dt.day_name()`, datetime subtraction + `.dt.days` for elapsed-time features, and treating `NaT`-derived `NaN`s as meaningful ("never happened") rather than missing data to patch over

**Why it matters for the project:** combining your own application data with an external bureau file, and turning raw dates into "days since X" features, is exactly how a real credit risk dataset gets built before modeling — this phase is the bridge between "data I have" and "data I need."

**What's next:** Phase 7, whenever you're ready — let me know the topics you want covered.